# **Foundry Memory**

[Reference](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/what-is-memory)

### **Step 1: Configuration**

In [1]:
AZURE_AI_PROJECT_ENDPOINT="https://resource-ms-foundary.services.ai.azure.com/api/projects/proj-ms-foundary"
CHAT_MODEL_DEPLOYMENT_NAME="gpt-4o-mini"
EMBD_MODEL_DEPLOYMENT_NAME="text-embedding-3-small"
STORE_NAME = "lc-integration-test-store"

### **Step 2: Install required libraries**

In [2]:
# ! pip install "azure-ai-projects>=2.0.0b4"

In [3]:
! pip show azure-ai-projects

Name: azure-ai-projects
Version: 2.0.1
Summary: Microsoft Corporation Azure AI Projects Client Library for Python
Home-page: 
Author: 
Author-email: Microsoft Corporation <azpysdkhelp@microsoft.com>
License-Expression: MIT
Location: /Users/kanavbansal/Developer/.env_jupyter/lib/python3.13/site-packages
Requires: azure-core, azure-identity, azure-storage-blob, isodate, openai, typing-extensions
Required-by: langchain-azure-ai


In [4]:
! pip show azure.identity

Name: azure-identity
Version: 1.25.2
Summary: Microsoft Azure Identity Library for Python
Home-page: 
Author: 
Author-email: Microsoft Corporation <azpysdkhelp@microsoft.com>
License-Expression: MIT
Location: /Users/kanavbansal/Developer/.env_jupyter/lib/python3.13/site-packages
Requires: azure-core, cryptography, msal, msal-extensions, typing-extensions
Required-by: azure-ai-projects, azure-monitor-opentelemetry-exporter, langchain-azure-ai


In [7]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    user_agent="langchain-azure-ai",
)

if not hasattr(client, "beta") or not hasattr(client.beta, "memory_stores"):
    raise ImportError(
        "Import Error"
)
else:
    # List all memory stores
    stores_list = list(client.beta.memory_stores.list())
    print(f"Found {len(stores_list)} memory stores")
    for store in stores_list:
        print(f"- {store.name} ({store.description})")

Found 0 memory stores


#### **Clean up memory store (If needed)**

In [8]:
# # Delete the entire memory store
# memory_store_name="lc-integration-test-store"
# delete_response = client.beta.memory_stores.delete(memory_store_name)
# print(f"Deleted memory store: {delete_response.name}")

In [9]:
# delete_response

### **Step 3: Importing Libraries and Setup**

In [10]:
# Importing libraries
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
	MemoryStoreDefaultDefinition,
	MemoryStoreDefaultOptions,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential

In [11]:
# Quick setup
endpoint = AZURE_AI_PROJECT_ENDPOINT
credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=endpoint, credential=credential)
store_name = STORE_NAME

In [12]:
# Init chat model
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel

llm = AzureAIOpenAIApiChatModel(
  project_endpoint=endpoint,
  credential=credential,   
  model=CHAT_MODEL_DEPLOYMENT_NAME,
)

llm.invoke("hi").text

'Hello! How can I assist you today?'

### **Step 4: Create a Memory Store**

Create a dedicated memory store for each agent to establish clear boundaries for memory access and optimization. When you create a memory store, specify the chat model and embedding model deployments that process your memory content.

In [13]:
try:
    store = client.beta.memory_stores.get(store_name)
    print(f"✓ Memory store '{store_name}' already exists")
except ResourceNotFoundError:
    print(f"Creating memory store '{store_name}'...")
    
    # Specify memory store options
    options = MemoryStoreDefaultOptions(
        user_profile_enabled=True,
        chat_summary_enabled=True,
        user_profile_details="Avoid irrelevant or sensitive data, such as age, financials, precise location, and credentials"
    )
    
    # Create memory store
    definition = MemoryStoreDefaultDefinition(
        chat_model=CHAT_MODEL_DEPLOYMENT_NAME, 			   # Put your LLM model
        embedding_model=EMBD_MODEL_DEPLOYMENT_NAME,	       # Put your emebddings model
        options=options
    )

    # Init the client
    client = AIProjectClient(
        endpoint=AZURE_AI_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
        user_agent="langchain-azure-ai",
    )

    # Init the memory store
    store = client.beta.memory_stores.create(
        name=store_name,
        description="Long-term memory store",
        definition=definition,
    )
    print(f"✓ Memory store '{store.name}' created successfully")

Creating memory store 'lc-integration-test-store'...
✓ Memory store 'lc-integration-test-store' created successfully


In [14]:
print(f"Name: {store.name}")
print(f"Created At: {store.created_at}")
print(f"Definition: {store.definition}")
print(f"Description: {store.description}")
print(f"Id: {store.id}")

Name: lc-integration-test-store
Created At: 2026-03-15 22:05:39+00:00
Definition: {'kind': 'default', 'chat_model': 'gpt-4o-mini', 'embedding_model': 'text-embedding-3-small', 'options': {'user_profile_enabled': True, 'user_profile_details': 'Avoid irrelevant or sensitive data, such as age, financials, precise location, and credentials', 'chat_summary_enabled': True}}
Description: Long-term memory store
Id: memstore_4d4eb20c090d6592006GrPPfdYFG4AmdOAmBuBCb862UQzrY4e


In [15]:
store.as_dict()

{'object': 'memory_store',
 'id': 'memstore_4d4eb20c090d6592006GrPPfdYFG4AmdOAmBuBCb862UQzrY4e',
 'created_at': 1773612339,
 'updated_at': 1773612339,
 'name': 'lc-integration-test-store',
 'description': 'Long-term memory store',
 'metadata': {},
 'definition': {'kind': 'default',
  'chat_model': 'gpt-4o-mini',
  'embedding_model': 'text-embedding-3-small',
  'options': {'user_profile_enabled': True,
   'user_profile_details': 'Avoid irrelevant or sensitive data, such as age, financials, precise location, and credentials',
   'chat_summary_enabled': True}}}

#### **Update memory store**

Update memory store properties, such as description or metadata, to better manage memory stores.

In [16]:
# Update memory store properties
store = client.beta.memory_stores.update(
    name=store_name,
    description="Long-Term Memory Store"
)

store.as_dict()

{'object': 'memory_store',
 'id': 'memstore_4d4eb20c090d6592006GrPPfdYFG4AmdOAmBuBCb862UQzrY4e',
 'created_at': 1773612339,
 'updated_at': 1773612341,
 'name': 'lc-integration-test-store',
 'description': 'Long-Term Memory Store',
 'metadata': {},
 'definition': {'kind': 'default',
  'chat_model': 'gpt-4o-mini',
  'embedding_model': 'text-embedding-3-small',
  'options': {'user_profile_enabled': True,
   'user_profile_details': 'Avoid irrelevant or sensitive data, such as age, financials, precise location, and credentials',
   'chat_summary_enabled': True}}}

### **Step 5: Create the Chat Message History Retriever**

The `scope` parameter controls how memory is partitioned. Each `scope` in the memory store keeps an isolated collection of memory items. For example, if you create a customer support agent with memory, each customer should have their own individual memory.

As a developer, you choose the key used to store and retrieve memory items. You can pass a static value, such as a universally unique identifier (UUID) or another stable identifier from your system.

Alternatively, when you specify `{{$userId}}` as the `scope`, the system automatically extracts the tenant ID (TID) and object ID (OID) from the request authentication header. This approach gives each authenticated user their own isolated memory partition, eliminating the need to manage identifiers manually.



In [17]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_azure_ai.chat_message_histories import AzureAIMemoryChatMessageHistory
from langchain_azure_ai.retrievers import AzureAIMemoryRetriever

_session_histories: dict[tuple[str, str], AzureAIMemoryChatMessageHistory] = {}

def base_factory(session_id: str):
    return InMemoryChatMessageHistory()

def get_session_history(user_id: str, session_id: str) -> AzureAIMemoryChatMessageHistory:
    """Get or create a session history for a user and session.
    
    Args:
        user_id: Stable user identifier (used as scope in Foundry Memory)
        session_id: Ephemeral session identifier
        
    Returns:
        AzureAIMemoryChatMessageHistory instance
    """
    cache_key = (user_id, session_id)

    if cache_key not in _session_histories:
        _session_histories[cache_key] = AzureAIMemoryChatMessageHistory(
            project_endpoint=endpoint,
            credential=credential,
            store_name=store_name,
            scope=user_id,
            session_id=session_id,
            base_history_factory=base_factory,
            update_delay=0,  # TEST MODE: process updates immediately (default ~30s)
        )
    return _session_histories[cache_key]


def get_foundry_retriever(user_id: str, session_id: str) -> AzureAIMemoryRetriever:
    """Get a retriever tied to the cached session history.
    
    This preserves incremental search state across turns.
    
    Args:
        user_id: Stable user identifier
        session_id: Ephemeral session identifier
        
    Returns:
        AzureAIMemoryRetriever instance
    """
    return get_session_history(user_id, session_id).get_retriever(k=5)

### **Step 6: Compose Chat Message History Runnable**

In [18]:
from typing import Any
import os

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import ConfigurableFieldSpec, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful and concise. Use prior memories when relevant."),
        MessagesPlaceholder("history"),
        ("system", "Memories:\n{memories}"),
        ("human", "{question}"),
    ]
)


def chain_for_session(user_id: str, session_id: str) -> RunnableWithMessageHistory:
    """Create a chain with message history for a specific user and session.
    
    Args:
        user_id: Stable user identifier
        session_id: Ephemeral session identifier
        
    Returns:
        Runnable chain with message history
    """
    retriever = get_foundry_retriever(user_id, session_id)

    def format_memories(x: dict[str, Any]) -> str:
        """Retrieve and format memories as text."""
        docs = retriever.invoke(x["question"])
        return (
            "\n".join([doc.page_content for doc in docs])
            if docs
            else "No relevant memories found."
        )
    
    # Use RunnablePassthrough.assign to add memories to the input dict
    # RunnableWithMessageHistory will inject history automatically
    chain = RunnablePassthrough.assign(memories=format_memories) | prompt | llm

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_session_history,
        input_messages_key="question",
        history_messages_key="history",
        history_factory_config=[
            ConfigurableFieldSpec(
                id="user_id",
                annotation=str,
                name="User ID",
                description="Unique identifier for the user.",
                default="",
                is_shared=True,
            ),
            ConfigurableFieldSpec(
                id="session_id",
                annotation=str,
                name="Session ID",
                description="Unique identifier for the session.",
                default="",
                is_shared=True,
            ),
        ],
    )
    return chain_with_history

### **Step 7: Run a Simple Scenario**

In [31]:
user_id = "user_001"
session_id = "session_2026_02_10_001"

In [32]:
chain = chain_for_session(user_id, session_id)

In [33]:
# Session A: seed preferences (long-term memory extraction happens async)
print(
	"\n=== Turn 1 (Session A): Introduce a preference "
	"(will be extracted into long-term memory) ==="
)

r1 = chain.invoke(
	{"question": "Hi! Call me JT. I prefer dark roast coffee and budget trips."},
	config={"configurable": {"user_id": user_id, "session_id": session_id}},
)
print()
print("ASSISTANT:", r1.content[0]["text"])


=== Turn 1 (Session A): Introduce a preference (will be extracted into long-term memory) ===

ASSISTANT: Hi again, JT! Dark roast coffee and budget trips it is. What’s on your mind today?


In [34]:
print("\n=== Turn 2 (Session A): Add another preference ===")

r2 = chain.invoke(
	{
		"question": "Also, I usually drink green tea in the afternoon "
		"and I like staying in hostels."
	},
	config={"configurable": {"user_id": user_id, "session_id": session_id}},
)
print()
print("ASSISTANT:", r2.content[0]["text"])


=== Turn 2 (Session A): Add another preference ===

ASSISTANT: Got it, JT! Dark roast coffee in the morning, green tea in the afternoon, and a love for hostels—sounds like a solid routine. What travel plans or tips are you looking for?


### **Step 8 - Run a Cross-session Scenario**

In [35]:
# Cross-session test: same user_id, new session_id
user_id = "user_001"
session_id_b = "session_2026_02_10_002"

chain_b = chain_for_session(user_id, session_id_b)

In [36]:
print("\n=== Turn 3 (Session B): New session should recall coffee preference ===")

r4 = chain_b.invoke(
	{"question": "Remind me of my coffee preference and travel style."},
	config={"configurable": {"user_id": user_id, "session_id": session_id_b}},
)
print()
print("ASSISTANT:", r4.content[0]["text"])


=== Turn 3 (Session B): New session should recall coffee preference ===

ASSISTANT: I currently don't have details about your coffee preference or travel style saved. If you provide that information, I can help you keep track of it!


In [37]:
print("\n=== Turn 4 (Session B): Retrieve another preference ===")

r5 = chain_b.invoke(
	{
		"question": "What do I usually drink in the afternoon, "
		"and where do I like to stay?"
	},
	config={"configurable": {"user_id": user_id, "session_id": session_id_b}},
)
print()
print("ASSISTANT:", r5.content[0]["text"])


=== Turn 4 (Session B): Retrieve another preference ===

ASSISTANT: I don't have your specific preferences stored right now. If you let me know what you usually drink in the afternoon and your preferred accommodations while traveling, I can remember that for future reference!


### **Clean up memory resources**

In [38]:
user_id="user_001"

result = client.beta.memory_stores.delete_scope(
    name=store_name, 
    scope=user_id
)

print(
	f"Deleted {getattr(result, 'deleted_count', 'all')} memories "
	f"for scope '{user_id}'."
)

Deleted all memories for scope 'user_001'.


## **Source Code Changes**

client.memory_stores has to be updated to client.beta.memory_stores in following files:
- chat_message_histories/azure_ai_memory.py
- retrivers/azure_ai_memory_retriever.py


**Path:** langchain_azure_ai > chat_message_histories > azure_ai_memory.py

```python
    class AzureAIMemoryChatMessageHistory(BaseChatMessageHistory):
        client = AIProjectClient(
            endpoint=self._project_endpoint,
            credential=cred,
            user_agent="langchain-azure-ai",
        )

        # if not hasattr(client, "memory_stores"): <--- This has to be updated with the following if logic
        if not hasattr(client, "beta") or not hasattr(client.beta, "memory_stores"):
            raise ImportError(
                "AzureAIMemoryChatMessageHistory requires azure-ai-projects>=2.0.0b4. "
                "Install the v2 extra: pip install 'langchain-azure-ai[v2]'"
            )
        self._client = client

        self._store = store_name
        self._scope = scope
        self._session_id = session_id
        self._base = base_history_factory(session_id)
        self._update_delay = update_delay
        self._role_mapper = role_mapper
        self._previous_update_id: Optional[str] = None  # advanced incremental updates
```

```python
def add_message():
    try:
        item = self._map_lc_message_to_foundry_item(message)
        # self._client.memory_stores.....()
        self._client.beta.memory_stores.begin_update_memories(  # type: ignore[attr-defined]
            name=self._store,
            scope=self._scope,
            items=[item],
            update_delay=self._update_delay,
            # previous_update_id=self._previous_update_id,  # optional
        )
        # non-blocking: do NOT poll; let the service extract after update_delay
    except Exception as e:
        # Intentionally swallow to avoid breaking chat flow; log for observability
        logger.warning(
            f"Failed to update Foundry Memory for message: {e}",
            exc_info=False,
        )

```

**Path:** langchain_azure_ai > retrivers > azure_ai_memory_retriever.py
```python
# Use previous_search_id only for history-bound (incremental) retrieval
result = self.client.beta.memory_stores.search_memories(
    name=self.store_name,
    scope=self.scope,
    items=items,
    previous_search_id=self._previous_search_id if incremental_search else None,
    options=MemorySearchOptions(max_memories=self.k),
)
```